<center>
<img src="https://upload.wikimedia.org/wikipedia/fr/thumb/1/1d/Logo_T%C3%A9l%C3%A9com_SudParis.svg/1014px-Logo_T%C3%A9l%C3%A9com_SudParis.svg.png" width="10%" />
</center>

<center> <h2> NET 4103/7431 Complex Network </h2> </center>

<center> <h3> Vincent Gauthier (vincent.gauthier@telecom-sudparis.eu) </h3> </center>

### Note
Avant de commencer les exercices, assurez-vous que tout fonctionne comme prévu. Tout d'abord, le redémarrage du kernel **(dans la barre de menus, sélectionnez le kernel $\rightarrow$ Restart)**.

Assurez-vous que vous remplir les célluler aux endroits marquer «YOUR CODE HERE». 

Veuillez supprimer les ligne «raise NotImplementedError()» dans toutes les cellules auxquelles vous avez répondu, ainsi que votre nom et prénom ci-dessous:

In [ ]:
NOM = "XXX"
PRENOM = "XXX"

---

In [ ]:
import duckdb
import pandas as pd
import tqdm as notebook_tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse._csr import csr_matrix
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from datasets.arrow_dataset import Dataset

In [ ]:
class DBStore:
    def __init__(self, db_path: str = "./database-p3.db", embedding_size:int = 384, max_feature:int = 20000) -> self:
        """
        Initialize 

        Args:
            db_path: Path to database file
        """
        self.conn = duckdb.connect(db_path)
        self.embedding_size = embedding_size
        self.max_feature = max_feature
        self.model = SentenceTransformer("all-MiniLM-L6-v2")   # 384 dimensions
        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            min_df=2,                             # ignore terms appearing in fewer than 2 passages
            max_df=0.5,                           # ignore terms appearing in more than 50% of passages
            max_features=self.max_feature,        # keep only the top N terms by corpus frequency
        )

    def _create_table(self) -> None:
        """Create the table if it doesn't exist"""
        try:
            # Use IF NOT EXISTS to avoid errors when table already exists
            create_table_query = f"""
            CREATE TABLE IF NOT EXISTS content (
                doc_id UInt32,
                content String
            )
            """
            self.conn.execute(create_table_query)
        except Exception as e:
            print(f"Warning when creating table: {str(e)}")
            print("Continuing with existing table...")
    
    def _create_index(self) -> None:
        try:
            query = """
            -- Install extension for Vector Similarity Search 
            INSTALL vss;
            -- Load extension
            LOAD vss;
            SET hnsw_enable_experimental_persistence = true;
            -- drop index if exist
            DROP INDEX IF EXISTS idx;
            -- Create an HNSW (Hiearchical Navigable Small World) index on the column 'embedding'
            CREATE INDEX idx ON embedding USING HNSW (embedding)
            WITH (metric = 'cosine');
            """
            self.conn.execute(query)
        except Exception as e:
            print(f"Warning when creating index: {str(e)}")

    def _generate_embedding(self, df: pd.DataFrame) -> None:
        try:
            # Use IF NOT EXISTS to avoid errors when table already exists
            create_table_query = f"""
            CREATE TABLE IF NOT EXISTS embedding (
                doc_id UInt32,
                embedding FLOAT[{self.embedding_size}]
            )
            """
            self.conn.execute(create_table_query)
            # create index
            self._create_index()
            # generate embedding
            embeddings = self.model.encode(
                df["passage"].tolist(),
                batch_size=64,
                show_progress_bar=True,
                normalize_embeddings=True,   # useful for cosine similarity
            ).tolist()
        
            df_emb = pd.DataFrame({"doc_id": df["id"].tolist(), "embedding": embeddings})
            query ="""
            DELETE FROM embedding;
            INSERT INTO embedding (doc_id, embedding)
            SELECT 
                doc_id,
                embedding
            FROM df_emb
            """
            self.conn.execute(query)
        except Exception as e:
            print(f"Warning when generating embedding: {str(e)}")

    def _generate_vocab(self, corpus: pd.DataFrame):
        docs = list(corpus["passage"])
        try:
            self.conn.execute("""
            CREATE OR REPLACE TABLE vocab(
                term_id UInt32, 
                term String
            );
            """)
            X = self.vectorizer.fit_transform(docs)
            vocab_df = pd.DataFrame({
                "term_id": range(len(self.vectorizer.get_feature_names_out())),
                "term": self.vectorizer.get_feature_names_out(),
            })
            self.conn.execute("""
            DELETE FROM vocab;
            INSERT INTO vocab SELECT * FROM vocab_df
            """)
            self._generate_tfidf(X)
        except Exception as e:
            print(f"Warning when generating vocab: {str(e)}")

    def _generate_tfidf(self, tfidf: csr_matrix):
        try:
            self.conn.execute("""
            CREATE OR REPLACE TABLE tfidf(
                doc_id UInt32, 
                term_id UInt32, 
                w FLOAT
            );
            """)
            coo = tfidf.tocoo()
            tf_df = pd.DataFrame({"doc_id": coo.row, "term_id": coo.col, "w": coo.data.astype("float32")})
            self.conn.execute("""
            DELETE FROM tfidf;
            INSERT INTO tfidf SELECT * FROM tf_df;
            """)
        except Exception as e:
            print(f"Warning when generating tf-idf: {str(e)}")

    def search_r1(self, query: str, topk: int = 5) -> pd.DataFrame:
        try:
            query_embedding = self.model.encode([query], normalize_embeddings=True).tolist()[0]
    
            db_query = f"""
            SELECT 
                doc_id, 
                c.content,
                array_distance(e.embedding, {query_embedding}::FLOAT[{self.embedding_size}]) AS distance,
                RANK() OVER (ORDER BY distance) AS rank
            FROM embedding e
            JOIN content c USING (doc_id)
            ORDER BY array_distance(embedding, {query_embedding}::FLOAT[{self.embedding_size}])
            LIMIT {topk};
            """
            return self.conn.sql(db_query).df() 
        except Exception as e:
            print(f"Warning when doing query: {str(e)}")

    def search_r2(self, question, topk=5) -> pd.DataFrame: 
        q = self.vectorizer.transform([question]).tocoo()
        qdf = pd.DataFrame({"term_id": q.col, "qw": q.data})
        return self.conn.execute(f"""
        SELECT
            p.doc_id,
            p.content, 
            SUM(t.w * qdf.qw) AS score,
            RANK() OVER (ORDER BY score DESC) AS rank
        FROM tfidf t
        JOIN qdf USING (term_id)
        JOIN content p USING (doc_id)
        GROUP BY p.doc_id, p.content
        ORDER BY score DESC
        LIMIT ?
        """, [topk]).df()
        
    def load_dataset(self, dataset: Dataset) -> None:
        try:
            df = dataset.to_pandas()
            self._create_table()            
            # Load in database
            query ="""
            DELETE FROM content;
            INSERT INTO content (doc_id, content)
            SELECT 
                id AS doc_id,
                passage AS content,
            FROM df
            """
            self.conn.execute(query)
            self._generate_embedding(df)
            self._generate_vocab(df)
            
        except Exception as e:
            print(f"Warning when loading dataset: {str(e)}")
    
    def close(self) -> None:
        self.conn.close()

In [ ]:
corpus = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus")["passages"]
qa = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer")["test"]

In [ ]:
db = DBStore()

db.load_dataset(corpus)

question = 'Was Abraham Lincoln the sixteenth President of the United States?'
results_r1 = db.search_r1(question)
results_r2 = db.search_r2(question)

db.close()

In [ ]:
def reciprocal_rank_fusion(r1: pd.DataFrame, r2: pd.DataFrame, k=60):
    df = r1.merge(r2, on="doc_id", how="outer", suffixes=("_x", "_y"))
    df["content"] = df["content_x"].fillna(df["content_y"])
    df = df.drop(columns=["content_x", "content_y", "score", "distance"])
    df = df.rename(columns={"rank_x": "r1", "rank_y": "r2"})
    df["rrf"] = 0.0
    retrivers = ["r1","r2"]
    for retriver in retrivers:
        df["score_" + retriver] =  1/(k + df[retriver]).fillna(float("inf"))
        df["rrf"] += df["score_" + retriver]
    df = df.sort_values("rrf", ascending=False)
    df["final_rank"] = df["rrf"].rank(ascending=False, method="min").astype(int)
    return df

In [ ]:
reciprocal_rank_fusion(results_r1, results_r2)